In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# Load Silver table
# ---------------------------------------------------------

silver = spark.table("financial_balance_sheets_silver")

print("Silver rows:", silver.count())
print("Silver columns:", len(silver.columns))

display(silver.limit(10))

In [0]:
# ---------------------------------------------------------
# Silver -> Gold
# Curated analytical financial positions in EUR
# ---------------------------------------------------------

gold = (
    silver
    .filter(F.col("unit") == "MIO_EUR")
    .select(
        "reference_year",
        "geo",
        "geo_label",
        "co_nco",
        "co_nco_label",
        "sector",
        "sector_label",
        "finpos",
        "finpos_label",
        "na_item",
        "na_item_label",
        "value",
        "status",
        "observation_key"
    )
)

print("Gold rows:", gold.count())
print("Gold columns:", len(gold.columns))

display(gold.limit(20))

In [0]:
# ---------------------------------------------------------
# Save Gold Delta table
# ---------------------------------------------------------

table_name = "financial_balance_sheets_gold"

(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

print(
    f"Gold table created successfully: {table_name}"
)

In [0]:
# ---------------------------------------------------------
# Example business view
# Financial positions by sector and position
# ---------------------------------------------------------

gold_summary = (
    gold
    .groupBy(
        "reference_year",
        "sector",
        "sector_label",
        "finpos",
        "finpos_label"
    )
    .agg(
        F.sum("value").alias("total_value_mio_eur")
    )
    .orderBy(
        "sector",
        "finpos"
    )
)

display(gold_summary)